## OpenAI 연동 기본 예제

- `ChatOpenAI`로 OpenAI API에 연결하고, Prompt → LLM → OutputParser를 LCEL로 연결합니다.
- 사전 준비: `.env` 파일에 `OPENAI_API_KEY` 설정, `uv add langchain-openai python-dotenv`

In [1]:
import os
from dotenv import load_dotenv

#load_dotenv(dotenv_path='.env')
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# 키 값은 출력하지 않고 설정 여부만 확인
if not OPENAI_API_KEY:
    raise RuntimeError(".env 파일에 OPENAI_API_KEY 가 없습니다.")
print("OPENAI_API_KEY 설정 확인 완료")

OPENAI_API_KEY 설정 확인 완료


#### OpenAI API로 호출 가능한 모델 목록 확인

- `openai` SDK(`langchain-openai`가 내부적으로 사용)의 `client.models.list()`로 현재 API 키로 사용할 수 있는 모델을 조회합니다.
- 목록 조회는 토큰을 사용하지 않아 비용이 발생하지 않습니다.
- 계정/조직에 따라 목록이 다를 수 있으며, 채팅 모델 외에 임베딩, 이미지, 음성 모델도 함께 나옵니다.

In [ ]:
# openai SDK 불러오기 (langchain-openai가 내부적으로 사용하는 SDK)
from openai import OpenAI

# API 키로 클라이언트 생성 (OPENAI_API_KEY는 load_dotenv()로 .env에서 읽어 둔 값)
client = OpenAI(api_key=OPENAI_API_KEY)

# 호출 가능한 모델 목록 조회 (id 기준 오름차순 정렬)
models = sorted(client.models.list().data, key=lambda m: m.id)
print(f"조회된 모델 수: {len(models)}\n")
for m in models:
    print(f"{m.id:<45} {m.owned_by}")

In [2]:
from langchain_core.prompts import ChatPromptTemplate

# prompt
prompt = ChatPromptTemplate.from_messages(
    [ ("system", "당신은 개발자입니다.") , 
     ("user", "{input}") ]
)
print(type(prompt))
print(prompt)

c:\myclass\ai_langchain\mylangchin-uv-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<class 'langchain_core.prompts.chat.ChatPromptTemplate'>
input_variables=['input'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='당신은 개발자입니다.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})]


In [3]:
prompt_text = prompt.format(input="파이썬은 무엇인가요? 자세하게 설명해주세요")
print(type(prompt_text))
print(prompt_text)

<class 'str'>
System: 당신은 개발자입니다.
Human: 파이썬은 무엇인가요? 자세하게 설명해주세요


In [4]:
from langchain_openai import ChatOpenAI

# 모델명은 상수로 분리
OPENAI_MODEL = "gpt-4o-mini"

# OpenAI API를 사용하는 ChatOpenAI 인스턴스 생성 (base_url 생략 시 OpenAI 기본 엔드포인트 사용)
llm = ChatOpenAI(
    api_key=OPENAI_API_KEY,
    model=OPENAI_MODEL,
    temperature=0.7
)
print(type(llm))
print(llm.model_name)

<class 'langchain_openai.chat_models.base.ChatOpenAI'>
gpt-4o-mini


In [6]:
try:
    response = llm.invoke(prompt_text)
    print(type(response))
    print("응답:", response.content)
except Exception as e:
    print(f"오류 발생: {e}")

<class 'langchain_core.messages.ai.AIMessage'>
응답: 파이썬(Python)은 고급 프로그래밍 언어로, 다양한 용도로 사용됩니다. 1991년 귀도 반 로썸(Guido van Rossum)에 의해 처음 출시되었으며, 인기가 높아지면서 많은 분야에서 활용되고 있습니다. 다음은 파이썬의 주요 특징과 장점에 대한 자세한 설명입니다.

### 1. **문법의 간결성**
   - 파이썬은 읽기 쉬운 문법을 가지고 있어, 코드가 간결하고 명확합니다. 이는 개발자들이 코드를 쉽게 이해하고 유지보수할 수 있도록 도와줍니다.
   - 예를 들어, 변수 선언 시 타입을 명시할 필요가 없고, 들여쓰기를 통해 코드 블록을 구분합니다.

### 2. **다양한 라이브러리와 프레임워크**
   - 파이썬은 데이터 분석, 웹 개발, 인공지능, 기계 학습 등 다양한 분야에서 사용할 수 있는 풍부한 라이브러리와 프레임워크를 제공합니다. 
   - 예를 들어, 데이터 분석을 위한 `Pandas`, 웹 개발을 위한 `Django`와 `Flask`, 머신러닝을 위한 `TensorFlow`와 `scikit-learn` 등이 있습니다.

### 3. **다양한 플랫폼 지원**
   - 파이썬은 Windows, macOS, Linux 등 다양한 운영 체제에서 실행할 수 있어 플랫폼 독립성이 뛰어납니다.
   - 이는 개발자들이 다양한 환경에서 파이썬 코드를 실행할 수 있다는 장점을 제공합니다.

### 4. **객체 지향 프로그래밍**
   - 파이썬은 객체 지향 프로그래밍(OOP) 패러다임을 지원합니다. 클래스와 객체를 통해 코드의 재사용성과 확장성을 높일 수 있습니다.

### 5. **인터프리터 언어**
   - 파이썬은 인터프리터 언어로, 코드를 작성한 후 즉시 실행할 수 있어 개발 및 디버깅 과정이 용이합니다.
   - 이는 다른 컴파일 언어에 비해 빠른 프로토타이핑이 가능하다는 장점이 있습니다.

### 6. **강력한 커뮤니티**
   - 파이썬은 전 세

## LCEL(LangChain Expression Language)
### Prompt + LLM 모델 + OutputParser

In [ ]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

In [ ]:
chain = prompt | llm | output_parser
print(type(chain))
print(chain)

In [ ]:
response = chain.invoke({"input":"LangChain은 무엇인가요?"})

In [ ]:
print(type(response))
print(response)